# Stage 6 — A/B/C Variant Comparison (Trixie Mok resume)

**Represents 5 of the 20 required test cases** (this resume × 5 jobs). One of four resume-based batches (Trixie Mok, Alex Chen, John Doe, +1 more), 5 test cases each, totalling 20.

Real experiment (not hand-simulated): the same source resume, `Trixie_Mok_Resume_AI.pdf`, tailored against 5 different job postings from the running app's own job database, through three system variants:

- **A — Minimal LLM**: bare prompt, "Rewrite this resume for this job."
- **B — Simplified system**: "Extract relevant skills and tailor this resume to the job" — no schema, no anti-fabrication instructions.
- **C — Full system**: the actual deployed app logic (`JobPortalService._generate_tailored_resume`), which enforces a structured JSON schema, requires `evidence_used` for every claim, and explicitly instructs: *"Never invent credentials, employers, dates, metrics, skills, or experience."*

All three variants ran against the same live LLM (Hugging Face router, `meta-llama/Llama-3.3-70B-Instruct`) used by the app in `.env`.

**This is a retest.** The results below are current — regenerated after the team shipped a deterministic anti-fabrication check to System C (`_enforce_fidelity`), which was itself a fix for the fabrication gap this test originally found. The original pre-fix results are preserved as `*_PREFIX.json` for direct before/after comparison (see the Retest section near the end).

## 4 Evaluation Criteria

| Criterion | Question |
|---|---|
| **Grounding / No Fabrication** | Does the output rely only on facts present in the source resume, with no invented employers, dates, metrics, or skills? |
| **Personalization / Relevance** | Does it appropriately tailor emphasis to the target job, without force-fitting irrelevant claims? |
| **Correctness & Completeness** | Are the facts accurate, and is nothing materially important from the source dropped or altered? |
| **Clarity & Change Evidence** | Is the output well-structured, and does it explain what changed and why, traceable to source evidence? |

These map directly onto System C's real design — they aren't arbitrary; they're the exact properties C's prompt is engineered to guarantee.

In [1]:
import json
from pathlib import Path

HERE = Path(".")
naive = json.loads((HERE / "variant_comparison_results.json").read_text(encoding="utf-8"))
grounded = json.loads((HERE / "variant_comparison_results_grounded_judge.json").read_text(encoding="utf-8"))

CRITERIA = ["grounding", "personalization", "correctness", "clarity"]
print(f"Loaded {len(naive)} test cases (naive judge) and {len(grounded)} (grounding-gated judge)")

Loaded 5 test cases (naive judge) and 5 (grounding-gated judge)


## Round 1: Naive LLM-judge scoring

The judge was told to score 1-5 on each criterion, with no explicit instruction about how to weigh fabrication vs. fluency.

In [2]:
def print_table(results, score_key):
    header = f"{'Job':<22}{'Variant':<8}" + "".join(f"{c[:10]:<12}" for c in CRITERIA) + "avg"
    print(header)
    print("-" * len(header))
    totals = {v: {c: [] for c in CRITERIA} for v in "ABC"}
    for entry in results:
        scores_block = entry[score_key]
        for v in "ABC":
            row = scores_block[v]
            vals = [row[c]["score"] for c in CRITERIA]
            avg = sum(vals) / len(vals)
            for c, val in zip(CRITERIA, vals):
                totals[v][c].append(val)
            print(f"{entry['job_title'][:21]:<22}{v:<8}" + "".join(f"{val:<12}" for val in vals) + f"{avg:.2f}")
    print()
    print("OVERALL AVERAGES")
    for v in "ABC":
        per_c = {c: sum(totals[v][c]) / len(totals[v][c]) for c in CRITERIA}
        overall = sum(per_c.values()) / len(per_c)
        print(f"  {v}: " + ", ".join(f"{c}={val:.2f}" for c, val in per_c.items()) + f"  -> overall={overall:.2f}")

print_table(naive, "scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       2           4           3           4           3.25
Software Engineer     B       3           4           4           4           3.75
Software Engineer     C       5           3           5           3           4.00
Machine Learning      A       5           4           5           4           4.50
Machine Learning      B       4           5           4           5           4.50
Machine Learning      C       5           3           5           5           4.50
DevOps Engineer       A       2           4           3           4           3.25
DevOps Engineer       B       4           4           4           4           4.00
DevOps Engineer       C       5           3           5           3           4.00
Database Administrato A       4           5           5           4           4.50
Databa

### The surprise

**A and B (minimal / simplified) score highest overall, tied at 3.75 — ahead of C (full system) at 3.55.** That looks backwards — until you look at what A and B actually wrote.

In [3]:
entry = [e for e in naive if e["job_title"] == "Full Stack Developer"][0]
print("Job:", entry["job_title"], "@", entry["company_name"])
print()
print("--- Variant A output (excerpt) ---")
print(entry["outputs"]["A"][:600])
print()
print("--- Variant C output (summary + skills) ---")
c = entry["outputs"]["C"]
print("summary:", c["resume"].get("summary"))
print("skills:", c["resume"].get("skills"))
print()
print("--- Naive judge's reasoning ---")
for v in "AC":
    print(v, {k: entry["scores"][v][k]["reason"] for k in ("personalization", "clarity")})

Job: Full Stack Developer @ InnoWave Networks

--- Variant A output (excerpt) ---
Here is a rewritten version of the resume tailored to the job description:

**Trixie Grace Mok**
**Mobile:** [redacted-phone] / **Email:** [redacted-email] | **linkedin.com/in/trixie-mok**

**PROFESSIONAL SUMMARY**
Highly motivated and detail-oriented full-stack developer with experience in developing web applications and working with cutting-edge technologies. Proficient in MEAN stack (MongoDB, AngularJS, Express, Node.js) and familiar with Java, Postgres, Redis, RabbitMQ, and Elasticsearch. Excited about the prospect of transforming clinical trial workflow and contributing to the development

--- Variant C output (summary + skills) ---
summary: MSc Enterprise Artificial Intelligence candidate with a Management and Digital Innovation background and hands-on experience automating business processes with AI and low-code tools.
skills: ['Microsoft Excel', 'Microsoft Power Apps', 'Microsoft Power Automate', 

**Root cause**: Variant A fabricated `"Proficient in JavaScript, Node.js, HTML, CSS, MongoDB, PostgreSQL"` — none of which appear anywhere in the source resume (which lists Excel, Power Apps, Power Automate, SQL, R, Python, Git). The naive judge rewarded this as well-tailored and clear, and penalized C for correctly declining to invent those skills for a role the candidate doesn't actually fit.

**This is the actual finding worth reporting**: a naive LLM-judge is biased toward confident, fluent fabrication over honest grounding — exactly the failure mode System C's anti-fabrication instructions exist to prevent. Scoring naively would make the *simplest* system look best, which is the wrong conclusion.

## Round 2: Grounding-gated judge (the fix)

Same three outputs, same criteria — but the judge prompt now includes a hard rule: any fabricated skill/employer/metric caps that variant's grounding score at 1 and every other score at 2, regardless of how fluent the writing is.

In [4]:
print_table(grounded, "grounded_judge_scores")

Job                   Variant grounding   personaliz  correctnes  clarity     avg
---------------------------------------------------------------------------------
Software Engineer     A       1           2           2           2           1.75
Software Engineer     B       1           2           2           2           1.75
Software Engineer     C       5           4           5           5           4.75
Machine Learning      A       5           4           5           4           4.50
Machine Learning      B       5           4           5           4           4.50
Machine Learning      C       5           4           5           4           4.50
DevOps Engineer       A       1           2           2           2           1.75
DevOps Engineer       B       1           2           2           2           1.75
DevOps Engineer       C       5           4           5           5           4.75
Database Administrato A       5           4           5           4           4.50
Databa

In [5]:
# Fabrications the grounding-gated judge actually caught, per variant
for entry in grounded:
    fab = entry["grounded_judge_scores"].get("fabrications", {})
    if any(fab.get(v) for v in "ABC"):
        print(entry["job_title"], "@", entry["company_name"])
        for v in "ABC":
            items = fab.get(v) or []
            if items:
                print(f"  {v}: {items}")
        print()

Software Engineer @ WestGate Networks
  A: ['Java (basic)', 'Operating Systems: Windows, Linux (basic)', 'Familiarity with cloud-based technologies and agile development methodologies', 'Microsoft Power Apps and Power Automate certification', 'Python programming certification', 'Data analysis and visualization using R and Python']
  B: ['Java (basic)']

DevOps Engineer @ CloudHarbor Labs
  A: ['C#', 'Cloud platforms: AWS, Azure', 'CI/CD tools: Jenkins, Chef', 'Database management: MySQL, MongoDB']
  B: ['Shell Scripting or PowerShell']

Database Administrator @ Solstice Digital
  A: ['']
  B: ['']
  C: ['']

Full Stack Developer @ InnoWave Networks
  A: ['MEAN stack (MongoDB, AngularJS, Express, Node.js)', 'Java, Postgres, Redis, RabbitMQ, Elasticsearch']
  B: ['MEAN stack (MongoDB, AngularJS, Express, Node.js)', 'Java, Postgres, Redis, RabbitMQ, Elasticsearch', 'experience with MongoDB']



## Retest: Before vs. After the Anti-Fabrication Fix

This is the direct retest of the failure case originally reported. The original run found C itself still fabricated 3 minor items on the hardest job (Full Stack Developer). The team then added a deterministic post-generation check (`_enforce_fidelity`) that strips any skill not textually present in the source resume. Below: the identical experiment, re-run against the fixed code, compared against the preserved pre-fix results (`*_PREFIX.json`).

In [6]:
prefix_grounded = json.loads((HERE / "variant_comparison_results_grounded_judge_PREFIX.json").read_text(encoding="utf-8"))

def overall_by_variant(results, score_key):
    out = {}
    for v in "ABC":
        vals = []
        for entry in results:
            row = entry[score_key][v]
            vals.append(sum(row[c]["score"] for c in CRITERIA) / len(CRITERIA))
        out[v] = sum(vals) / len(vals)
    return out

before = overall_by_variant(prefix_grounded, "grounded_judge_scores")
after = overall_by_variant(grounded, "grounded_judge_scores")

print(f"{'Variant':<20}{'Before fix':<14}{'After fix':<14}")
for v, label in zip("ABC", ["A - Minimal", "B - Simplified", "C - Full system"]):
    print(f"{label:<20}{before[v]:<14.2f}{after[v]:<14.2f}")

print()
def fab_jobs_count(results, variant):
    n = 0
    for entry in results:
        items = (entry["grounded_judge_scores"].get("fabrications", {}) or {}).get(variant) or []
        if any(items):
            n += 1
    return n

print(f"C fabrications (jobs affected, of {len(grounded)}):")
print(f"  Before fix: {fab_jobs_count(prefix_grounded, 'C')} of {len(prefix_grounded)}")
print(f"  After fix:  {fab_jobs_count(grounded, 'C')} of {len(grounded)}")

Variant             Before fix    After fix     
A - Minimal         3.40          2.85          
B - Simplified      3.40          2.85          
C - Full system     3.80          4.55          

C fabrications (jobs affected, of 5):
  Before fix: 1 of 5
  After fix:  0 of 5


## Findings

**1. Same outputs, different judge, different winner.**

Nothing about A, B, or C changed between Round 1 and Round 2 — only the judge's instructions changed. That alone was enough to flip which system "won." Lesson: how you grade the AI matters as much as how you build it.

**2. C didn't lose Round 1 because it was worse — it lost because it refused to lie.**

C was designed to never invent skills the candidate doesn't have. In Round 1, the judge saw that honesty as a weaker, less impressive resume compared to A/B's confident (but made-up) answers. So Round 1 wasn't really testing the resumes — it was exposing a flaw in the judge.

**3. Round 2 just added one rule to the judge, and that fixed the judge.**

Same A/B/C answers as before. The only change: tell the judge "any made-up fact caps your score, no matter how well-written it sounds." With that one rule, C won clearly.

**4. The fix to the *system* (not just the judge) closed the remaining gap.**

The original test found C still fabricated 3 minor items on the hardest-fit job (Full Stack Developer). The team added a deterministic post-generation check (`_enforce_fidelity`) that strips any skill not textually present in the source resume, replacing the LLM's own self-report. Retested on the identical job, C now shows zero fabrications. This is a real fix verified by a real retest, not just a judge adjustment.

**5. As a Stage 7 failure case, now closed out:**

| Step | What happened |
|---|---|
| Input | Naive judge prompt, scoring A/B/C on the 4 criteria above |
| Expected | C (the fabrication-resistant system) scores highest |
| Actual | A/B scored highest instead (Round 1, both before and after the system fix — this is a judge problem, not a system problem) |
| Likely cause | The judge had no instruction to penalize invented content, so it rewarded confident, fluent writing over truthfulness |
| Fix #1 (judge) | Added one hard rule to the judge prompt: any fabricated fact caps that variant's scores, regardless of how well-written it is |
| Retest #1 result | Ranking corrected to C highest (3.80 vs. A/B's 3.40); revealed C itself still had 1 fabrication case (Full Stack Developer, 3 minor items) |
| Fix #2 (system) | Added a deterministic post-generation check (`_enforce_fidelity`) that strips any skill not textually present in the source resume |
| Retest #2 result | C's score rose to 4.55 with zero fabrications across all 5 jobs, including the previously-failing Full Stack Developer case — **both issues are now closed** |